In [18]:
import random
import math

n = 50
partners = 20
ants = 20
iterations = 100

alpha = 1
beta = 2
rho = 0.5

cities = [(random.randint(1, 100), random.randint(1, 100))
          for i in range(n)]

depot = (50, 50)

def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

dist = [[distance(cities[i], cities[j]) for j in range(n)]
        for i in range(n)]

depot_dist = [distance(depot, cities[i]) for i in range(n)]

pheromone = [[1.0 for j in range(n)] for i in range(n)]
depot_pheromone = [1.0] * n

best_distance = float("inf")
best_routes = []

sizes = [3] * 10 + [2] * 10

for iteration in range(iterations):

    for ant in range(ants):

        unvisited = list(range(n))
        routes = []
        total = 0

        for size in sizes:

            route = []
            current = -1

            for step in range(size):

                choices = []
                weights = []

                for city in unvisited:

                    if current == -1:
                        d = depot_dist[city]
                        p = depot_pheromone[city]
                    else:
                        d = dist[current][city]
                        p = pheromone[current][city]

                    weight = (max(p, 0.000001)**alpha) * (1/(d+0.001))**beta

                    choices.append(city)
                    weights.append(weight)

                city = random.choices(choices, weights=weights)[0]

                route.append(city)
                unvisited.remove(city)

                if current == -1:
                    total += depot_dist[city]
                else:
                    total += dist[current][city]

                current = city

            total += depot_dist[current]
            routes.append(route)

        if total < best_distance:
            best_distance = total
            best_routes = [r[:] for r in routes]

        for i in range(n):
            for j in range(n):
                pheromone[i][j] = max(pheromone[i][j] * (1-rho), 0.000001)

            depot_pheromone[i] = max(depot_pheromone[i] * (1-rho), 0.000001)

        for route in routes:

            depot_pheromone[route[0]] += 100 / total
            depot_pheromone[route[-1]] += 100 / total

            for i in range(len(route)-1):
                a = route[i]
                b = route[i+1]

                pheromone[a][b] += 100 / total
                pheromone[b][a] += 100 / total

print("\nOPTIMIZED DELIVERY PLAN")

print("\nTotal Distance:", round(best_distance, 2))
print("Partners Used:", partners)
print("Locations Covered:", n)


OPTIMIZED DELIVERY PLAN

Total Distance: 1975.66
Partners Used: 20
Locations Covered: 50


In [17]:
import random

traffic = [40, 60, 30, 50]

n = 10
Pa = 0.25
iterations = 50
cycle_time = 120

nests = []

for i in range(n):
    nest = [random.random() for j in range(4)]
    nests.append(nest)


def get_green_time(nest):

    total = sum(nest)

    green = []

    for i in range(4):
        time = 10 + (nest[i] / total) * 80
        green.append(time)

    return green


def fitness(nest):

    green = get_green_time(nest)

    total_wait = 0

    for i in range(4):
        waiting = (cycle_time - green[i]) / 2
        total_wait += traffic[i] * waiting

    return total_wait


def levy_flight():

    beta = 1.5

    u = random.gauss(0, 1)
    v = random.gauss(0, 1)

    step = u / (abs(v) ** (1 / beta) + 0.0001)

    return step


for iteration in range(iterations):

    for i in range(n):

        new_nest = []
        
        for j in range(4):

            step = levy_flight()
            value = nests[i][j] + 0.01 * step
            new_nest.append(abs(value) + 0.1)

        k = random.randint(0, n - 1)

        if fitness(new_nest) < fitness(nests[k]):
            nests[k] = new_nest

    count = int(Pa * n)

    nests.sort(key=fitness, reverse=True)

    for i in range(count):
        nests[i] = [random.random() + 0.1 for j in range(4)]


best_nest = min(nests, key=fitness)
green = get_green_time(best_nest)

print("Optimized Traffic Signal Timings")

for i in range(4):
    print("Road", i+1, ":", round(green[i], 2), "sec")

print("Waiting Cost:", round(fitness(best_nest), 2))

Optimized Traffic Signal Timings
Road 1 : 10.66 sec
Road 2 : 28.54 sec
Road 3 : 10.34 sec
Road 4 : 70.46 sec
Waiting Cost: 7814.02
